In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import random
import zipfile
import shutil
from tqdm import tqdm
SEED = 42
N_TEST = 15000

In [2]:

# Đổi path này nếu tracks.csv của bạn nằm chỗ khác
tracks_csv_candidates = list(Path("../archive/fma_medium/fma_metadata").rglob("tracks.csv"))
print(tracks_csv_candidates)

tracks_csv = tracks_csv_candidates[0]

tracks = pd.read_csv(tracks_csv, index_col=0, header=[0, 1])

subset = tracks[("set", "subset")]

large_only_ids = subset[subset == "large"].index.astype(int).tolist()

print("large-only tracks:", len(large_only_ids))
print("example:", large_only_ids[:10])

[PosixPath('../archive/fma_medium/fma_metadata/tracks.csv')]
large-only tracks: 81574
example: [20, 26, 30, 46, 48, 135, 137, 138, 142, 144]


In [3]:
random.seed(SEED)

selected_ids = sorted(random.sample(large_only_ids, N_TEST))
selected_id_set = set(selected_ids)

print("selected:", len(selected_ids))
print("first 20:", selected_ids[:20])
print("last 20:", selected_ids[-20:])

selected: 15000
first 20: [26, 48, 146, 147, 158, 160, 161, 168, 171, 173, 185, 199, 250, 274, 275, 281, 293, 305, 309, 310]
last 20: [155051, 155054, 155083, 155087, 155116, 155122, 155166, 155199, 155204, 155216, 155217, 155222, 155224, 155225, 155231, 155238, 155256, 155261, 155303, 155312]


In [4]:
!mkdir -p ../archive/fma_large_zip
!wget -c https://os.unil.cloud.switch.ch/fma/fma_large.zip -P ../archive/fma_large_zip

--2026-06-25 16:30:49--  https://os.unil.cloud.switch.ch/fma/fma_large.zip
Resolving os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)... 2001:620:5ca1:201::214, 86.119.28.16
Connecting to os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)|2001:620:5ca1:201::214|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 100306112191 (93G) [application/zip]
Saving to: ‘../archive/fma_large_zip/fma_large.zip’

fma_large.zip         2%[                    ]   2.56G  --.-KB/s    in 27m 58s 

2026-06-25 16:58:55 (1.56 MB/s) - Read error at byte 2748681888/100306112191 (Connection timed out). Retrying.

--2026-06-25 16:58:56--  (try: 2)  https://os.unil.cloud.switch.ch/fma/fma_large.zip
Connecting to os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)|2001:620:5ca1:201::214|:443... failed: Network is unreachable.
Connecting to os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)|86.119.28.16|:443... connected.
HTTP request sent, awaiting response... 206 Partial Content
Length: 10030

In [ ]:
FMA_LARGE_ZIP_DIR = Path("../archive/fma_large_zip")
zip_candidates = sorted(FMA_LARGE_ZIP_DIR.glob("*.zip"))

print("Zip candidates:")
for p in zip_candidates:
    print(p, p.stat().st_size / (1024**3), "GB")

assert zip_candidates, f"Không tìm thấy file zip trong {FMA_LARGE_ZIP_DIR}"

ZIP_PATH = zip_candidates[0]
print("Using ZIP:", ZIP_PATH)

In [ ]:
def track_id_from_zip_name(name: str) -> int | None:
    """
    Convert zip member path like:
        fma_large/000/000020.mp3
    to:
        20
    """
    if not name.endswith(".mp3"):
        return None
    
    stem = Path(name).stem
    if not stem.isdigit():
        return None
    
    return int(stem)


selected_members = []

OUT_RAW_DIR = Path("../crop_data/test/real_6s")
OUT_RAW_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH) as z:
    names = z.namelist()
    
    for name in names:
        tid = track_id_from_zip_name(name)
        if tid in selected_id_set:
            selected_members.append(name)
    
    print("selected members found in zip:", len(selected_members))
    assert len(selected_members) == N_TEST, f"Expected {N_TEST}, found {len(selected_members)}"

    for name in tqdm(selected_members):
        tid = track_id_from_zip_name(name)
        out_path = OUT_RAW_DIR / f"{tid:06d}.mp3"

        # resume
        if out_path.exists() and out_path.stat().st_size > 1000:
            continue

        with z.open(name) as src, open(out_path, "wb") as dst:
            shutil.copyfileobj(src, dst)

print("Extracted mp3:", len(list(OUT_RAW_DIR.glob('*.mp3'))))
print("Output:", OUT_RAW_DIR)

In [ ]:
rows = []

for tid in selected_ids:
    filename = f"{tid:06d}.mp3"
    path = OUT_RAW_DIR / filename
    
    rows.append({
        "track_id": tid,
        "filename": filename,
        "path": str(path),
        "subset": "large",
        "split": "test",
        "fake_type": "Real",
        "label": 1,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    })


MANIFEST_PATH = Path("../crop_data/test/test_real_manifest.csv")
manifest = pd.DataFrame(rows)
manifest.to_csv(MANIFEST_PATH, index=False)

print("Manifest saved:", MANIFEST_PATH)
print(manifest["exists"].value_counts())
manifest.head()

In [ ]:
print("MP3 count:", len(list(OUT_RAW_DIR.glob("*.mp3"))))

total_size_gb = sum(p.stat().st_size for p in OUT_RAW_DIR.glob("*.mp3")) / (1024**3)
print("Total size GB:", round(total_size_gb, 2))

print("Sample files:")
for p in list(OUT_RAW_DIR.glob("*.mp3"))[:10]:
    print(p.name)